# 6 Grafiken

Grafiken zu Training und Auswertung von V1 mit Vega-Altair, alle im gleichen Stil:
1. Konvergenz des Trainings (`train_verlauf.csv`)
2. Precision und Recall pro Schwelle auf den Ausschnitten (`03_auswertung_ausschnitte.ipynb`)
3. Lagegenauigkeit: Modell gegen eine einzelne menschliche Digitalisierung (`05_stufe2.ipynb`)
4. Fehler von Stufe 2 pro Plan
5. 10 zufällige Heatmaps aus den Trainingsdaten und 10 aus der Inferenz

Ausgabe zusätzlich als PNG und HTML in `data/auswertung/grafiken/`.

In [ ]:
import base64
import io
import os
import random

import altair as alt
import numpy as np
import pandas as pd
import torch
from PIL import Image

from grenzpunkte import DATA, DATENSATZ, PunktDataset, device, heatmap, modell_laden

STUFE2 = "data/auswertung/stufe2_v1/ergebnisse_val.csv"
SCHWELLEN = "data/auswertung/stufe1_schwellen_val.csv"
AUSGABE = "data/auswertung/grafiken"
FARBEN = ["#2a78d6", "#eb6834", "#1baf7a", "#e34948"]  # blau, orange, grün, rot
os.makedirs(AUSGABE, exist_ok=True)

## Einheitlicher Stil
Ein Altair-Theme legt Schrift, Achsen, Raster und Farben einmal fest; alle Grafiken übernehmen es.

In [ ]:
@alt.theme.register("grenzpunkt", enable=True)
def stil():
    return {"config": {
        "font": "Helvetica", "background": "white", "view": {"stroke": None},
        "title": {"fontSize": 16, "anchor": "start", "subtitleColor": "#52514e", "subtitleFontSize": 12},
        "axis": {"labelFontSize": 12, "titleFontSize": 12, "titleColor": "#52514e", "gridColor": "#e5e4e0",
                 "domainColor": "#52514e", "tickColor": "#52514e"},
        "legend": {"labelFontSize": 12, "titleFontSize": 12, "orient": "top", "labelLimit": 0},
        "range": {"category": FARBEN},
    }}


def speichern(chart, name):
    chart.save(f"{AUSGABE}/{name}.png", scale_factor=2)
    chart.save(f"{AUSGABE}/{name}.html")
    return chart

## Konvergenz
Mittlerer Loss pro Epoche. Die gestrichelte Linie markiert die Epoche mit dem kleinsten Val-Loss (gespeichertes Modell).
Beim ersten Training wurde der Trainings-Loss nur alle 100 Batches protokolliert; `train_verlauf.csv` enthält dort das Mittel dieser Werte.

In [ ]:
verlauf = pd.read_csv("train_verlauf.csv")
df = verlauf.melt("epoche", var_name="Datensatz", value_name="Loss").replace(
    {"train_loss": "Training", "val_loss": "Validierung"})
beste = verlauf.nsmallest(1, "val_loss")

linien = alt.Chart(df).mark_line(point=True).encode(
    x=alt.X("epoche:Q", title="Epoche", scale=alt.Scale(domain=[1, verlauf["epoche"].max()])),
    y=alt.Y("Loss:Q", title="BCE-Loss (Mittel pro Epoche)", scale=alt.Scale(type="log", domain=[0.005, 0.3])),
    color=alt.Color("Datensatz:N", title=None))
markierung = alt.Chart(beste).mark_rule(strokeDash=[4, 4], color="#52514e").encode(x="epoche:Q")
text = alt.Chart(beste).mark_text(align="left", dx=6, dy=-120, color="#52514e").encode(
    x="epoche:Q", text=alt.value("gespeichertes Modell"))
speichern((linien + markierung + text).properties(
    width=600, height=320, title=alt.Title("Konvergenz des Trainings", subtitle="U-Net ResNet34")), "konvergenz")

## Precision und Recall pro Schwelle
Stufe 1 auf den val-Ausschnitten. Eine tiefe Schwelle findet mehr Punkte (Recall), gibt aber mehr Fehldetektionen aus (Precision).

In [ ]:
df = pd.read_csv(SCHWELLEN).rename(columns={"precision": "Precision", "recall": "Recall"})
df = df.melt("schwelle", ["Precision", "Recall"], var_name="Kennzahl", value_name="Wert")
chart = alt.Chart(df).mark_line(point=True).encode(
    x=alt.X("schwelle:Q", title="Schwelle (Heatmap-Wert)"),
    y=alt.Y("Wert:Q", title=None, scale=alt.Scale(zero=False)),
    color=alt.Color("Kennzahl:N", title=None))
speichern(chart.properties(width=600, height=320, title=alt.Title(
    "Precision und Recall", subtitle="Stufe 1 auf den val-Ausschnitten, Paar bis 8 px")), "precision_recall")

## Lagegenauigkeit
Abstand zum Label in Pixeln (600 DPI) für die Treffer von Stufe 2. Zum Vergleich: Eine einzelne menschliche Digitalisierung liegt `d12 / 2` vom Label entfernt, denn das Label ist das Mittel aus D1 und D2.

In [ ]:
plaene = pd.read_csv(f"{DATENSATZ}/plaene.csv").query("status == 'ok'").set_index("plan_id")
e = pd.read_csv(STUFE2)
e = e[(e["status"] == "treffer") & (e["plan"] != "R_103")]
cm_pro_px = plaene["bodenpixel_m"] * 100 * plaene["scan_dpi"] / 600
modell = e["fehler_cm"] / e["plan"].map(cm_pro_px).values
mensch = pd.concat([pd.read_csv(f"{DATENSATZ}/labels/{p}.csv")["d12_px"].dropna() / 2 * 600 / plaene.at[p, "scan_dpi"]
                    for p in e["plan"].unique()])
df = pd.concat([pd.DataFrame({"Abstand": modell, "Wer": f"Modell (Median {modell.median():.2f} px)"}),
                pd.DataFrame({"Abstand": mensch, "Wer": f"eine Digitalisierung (Median {mensch.median():.2f} px)"})])

chart = alt.Chart(df[df["Abstand"] <= 3]).mark_bar(opacity=0.7, binSpacing=0).encode(
    x=alt.X("Abstand:Q", bin=alt.Bin(step=0.1), title="Abstand zum Label [px, 600 DPI]"),
    y=alt.Y("count():Q", title="Anzahl Punkte", stack=None),
    color=alt.Color("Wer:N", title=None))
speichern(chart.properties(width=600, height=320, title=alt.Title(
    "Lagegenauigkeit", subtitle="Treffer in Stufe 2, 9 Pläne 1:1000 (val), 1 px ≈ 4.2 cm")), "lagegenauigkeit")

## Fehler von Stufe 2 pro Plan

In [ ]:
namen = {"fehldetektion": "Fehldetektion", "verpasst": "verpasst", "falsches_symbol": "falsches Symbol"}
e = pd.read_csv(STUFE2)
e = e[e["status"].isin(namen)]
df = e.assign(Fehler=e["status"].map(namen)).groupby(["plan", "Fehler"]).size().reset_index(name="Anzahl")
chart = alt.Chart(df).mark_bar().encode(
    y=alt.Y("plan:N", title=None), x=alt.X("Anzahl:Q", title="Anzahl PN-Punkte"),
    color=alt.Color("Fehler:N", title=None, sort=list(namen.values())))
speichern(chart.properties(width=600, height=300, title=alt.Title(
    "Fehler von Stufe 2 pro Plan", subtitle="val-Pläne mit PN, Schwelle 0.5, alle PN-Punkte gesucht")), "stufe2_fehler")

## Heatmaps
Die Heatmap wird rot ins Bild gemischt und als PNG direkt in die Grafik eingebettet (`mark_image`). Gezeigt werden nur Ausschnitte mit mindestens einem Label.

In [ ]:
def bild_url(bild, hm):
    """Bild mit roter Heatmap darüber als PNG-Data-URL."""
    a = np.clip(hm, 0, 1)[..., None] * 0.85
    mischung = (np.asarray(bild, float) * (1 - a) + np.array([227, 73, 72]) * a).astype(np.uint8)
    puffer = io.BytesIO()
    Image.fromarray(mischung).save(puffer, format="PNG")
    return "data:image/png;base64," + base64.b64encode(puffer.getvalue()).decode()


def bildraster(df, titel, untertitel, name):
    """df mit Spalten url und Titel: Bilder in Reihen zu fünf."""
    chart = alt.Chart(df).mark_image(width=170, height=170).encode(
        url="url:N", x=alt.value(85), y=alt.value(85)).properties(width=170, height=170)
    chart = chart.facet(facet=alt.Facet("Titel:N", title=None, header=alt.Header(labelFontSize=11)), columns=5)
    return speichern(chart.properties(title=alt.Title(titel, subtitle=untertitel)), name)

In [ ]:
random.seed(0)
train_ds = PunktDataset("train")
zeilen = []
for bild in random.sample(sorted(train_ds.punkte), 10):
    rgb = Image.open(f"{DATA}/bilder/{bild}.png").convert("RGB")
    zeilen.append({"url": bild_url(rgb, heatmap(train_ds.punkte[bild], 256, 256)[0].numpy()),
                   "Titel": f"{bild}, Labels: {len(train_ds.punkte[bild])}"})
bildraster(pd.DataFrame(zeilen), "Trainingsdaten: Ziel-Heatmap",
           "Gaussglocke σ = 2 px pro Label, 256 × 256 px bei 600 DPI", "heatmaps_training")

In [ ]:
val_ds = PunktDataset("val")
model = modell_laden()
zeilen = []
for bild in random.sample(sorted(val_ds.punkte), 10):
    X, _ = val_ds[list(val_ds.bilder).index(bild)]
    with torch.no_grad():
        hm = torch.sigmoid(model(X[None].to(device)))[0, 0].cpu().numpy()
    rgb = Image.open(f"{DATA}/bilder/{bild}.png").convert("RGB")
    zeilen.append({"url": bild_url(rgb, hm), "Titel": f"{bild}, Labels: {len(val_ds.punkte[bild])}"})
bildraster(pd.DataFrame(zeilen), "Inferenz: vorhergesagte Heatmap",
           "unet.pt auf zufälligen Validierungsausschnitten", "heatmaps_inferenz")